In [1]:
# CELL 1: Analyze character, word and token lengths of scheme features

import json
import pandas as pd
from pathlib import Path
import tiktoken

INPUT_FILE = "G:\\SEVA_Ai\\SEVA_Ai\\data\\Raw\\all_schemes\\all_schemes.json"

with open(INPUT_FILE, encoding="utf-8") as f:
    data = json.load(f)

schemes = data if isinstance(data, list) else data["schemes"]

encoder = tiktoken.get_encoding("cl100k_base")

features = [
    "brief_description", "details", "eligibility",
    "benefits", "exclusions", "application_process",
    "documents_required"
]

def get_text(value):
    if value is None:
        return ""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

records = []

for scheme in schemes:
    for feature in features:
        text = get_text(scheme.get(feature))

        if text.strip():
            records.append({
                "scheme_id": scheme["scheme_id"],
                "feature": feature,
                "characters": len(text),
                "words": len(text.split()),
                "tokens": len(encoder.encode(text))
            })

length_df = pd.DataFrame(records)

print("Total feature records:", len(length_df))
length_df.head()

Total feature records: 27305


,scheme_id,feature,characters,words,tokens
0,LBezF5gB5uF80_MGGY81,brief_description,230,33,41
1,LBezF5gB5uF80_MGGY81,details,635,102,132
2,LBezF5gB5uF80_MGGY81,eligibility,241,46,55
3,LBezF5gB5uF80_MGGY81,benefits,421,63,86
4,LBezF5gB5uF80_MGGY81,application_process,1497,185,362


In [2]:
# CELL 2: Calculate length distribution for each feature

feature_stats = length_df.groupby("feature").agg(
    records=("scheme_id", "count"),
    avg_tokens=("tokens", "mean"),
    median_tokens=("tokens", "median"),
    max_tokens=("tokens", "max"),
    min_tokens=("tokens", "min"),
    avg_words=("words", "mean")
).round(2)

display(feature_stats)

,records,avg_tokens,median_tokens,max_tokens,min_tokens,avg_words
feature,,,,,,
application_process,4428,347.50,276.0,2757,1,193.18
benefits,4423,137.74,81.0,7311,3,87.83
brief_description,4428,47.80,49.0,87,12,35.58
details,4424,145.16,111.0,1670,18,107.29
documents_required,4428,110.90,69.5,1858,1,65.23
eligibility,4424,138.37,103.0,1846,7,102.55
exclusions,750,78.28,45.0,917,1,57.25


In [3]:
# CELL 3: Analyze original FAQ question and answer lengths

faq_records = []

for scheme in schemes:
    for faq in scheme.get("faqs", []):
        if not isinstance(faq, dict):
            continue

        question = get_text(faq.get("question"))
        answer = get_text(faq.get("answer"))

        if question.strip() and answer.strip():
            faq_records.append({
                "scheme_id": scheme["scheme_id"],
                "question_tokens": len(encoder.encode(question)),
                "answer_tokens": len(encoder.encode(answer)),
                "total_tokens": len(encoder.encode(
                    question + "\n" + answer
                ))
            })

faq_df = pd.DataFrame(faq_records)

display(faq_df[[
    "question_tokens", "answer_tokens", "total_tokens"
]].describe().round(2))

,question_tokens,answer_tokens,total_tokens
count,48695.00,48695.00,48695.00
mean,12.75,29.49,42.26
std,5.96,23.17,24.79
min,3.00,1.00,7.00
25%,9.00,17.00,28.00
50%,11.00,25.00,37.00
75%,15.00,35.00,51.00
max,77.00,1805.00,1818.00


In [5]:
# CELL 4: Export chunking analysis results

OUTPUT = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\output\\chunking_analysis"
OUTPUT = Path(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)

feature_stats.to_csv(OUTPUT / "feature_length_analysis.csv")
faq_df.to_csv(OUTPUT / "faq_length_analysis.csv", index=False)

length_df.to_csv(OUTPUT / "scheme_feature_lengths.csv", index=False)

print("Chunking analysis saved successfully!")

Chunking analysis saved successfully!


In [6]:
# CELL 1: Identify long documents and token length distribution

feature_analysis = length_df.groupby("feature")["tokens"].agg(
    records="count",
    median="median",
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95),
    p99=lambda x: x.quantile(0.99),
    maximum="max"
).round(2)

for limit in [500, 800, 1000]:
    feature_analysis[f"above_{limit}"] = (
        length_df.groupby("feature")["tokens"]
        .apply(lambda x: (x > limit).sum())
    )

display(feature_analysis)

,records,median,p90,p95,p99,maximum,above_500,above_800,above_1000
feature,,,,,,,,,
application_process,4428,276.0,669.3,876.65,1234.46,2757,867,294,162
benefits,4423,81.0,304.0,430.90,853.68,7311,157,53,23
brief_description,4428,49.0,63.0,67.00,74.00,87,0,0,0
details,4424,111.0,256.0,339.00,642.62,1670,90,21,10
documents_required,4428,69.5,244.0,329.00,710.38,1858,88,28,16
eligibility,4424,103.0,273.0,369.70,669.54,1846,110,25,14
exclusions,750,45.0,181.0,258.55,423.00,917,5,1,0


In [7]:
# CELL 2: Analyze long FAQ answers

faq_analysis = faq_df["total_tokens"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99]
)

print("FAQ token percentiles:")
print(faq_analysis)

print("\nFAQs exceeding 300 tokens:",
      (faq_df["total_tokens"] > 300).sum())

print("FAQs exceeding 500 tokens:",
      (faq_df["total_tokens"] > 500).sum())

FAQ token percentiles:
0.50     37.0
0.75     51.0
0.90     66.0
0.95     79.0
0.99    127.0
Name: total_tokens, dtype: float64

FAQs exceeding 300 tokens: 22
FAQs exceeding 500 tokens: 5


Excellent. Now we have enough information to make an evidence-based chunking decision for SEVA_AI.

The most important finding is that most of your scheme fields are already small. Only a small percentage needs advanced chunking.
1. Analysis of our results
Application process
Highest priority for chunking
- Median: 276 tokens.
- P90: 669 tokens.
- P99: 1,234 tokens.
- 867 records exceed 800 tokens.
- 294 records exceed 1,000 tokens.
This field needs special treatment because application steps should remain logically ordered.

Other scheme features
Feature	Above 800 tokens	Recommendation
Benefits	157	Split long records
Eligibility	110	Split long records
Documents required	88	Preserve lists
Details	90	Split long records
Exclusions	5	Mostly keep intact
Brief description	0	Keep intact


FAQs
Keep atomic by default
- Median: 37 tokens.
- P95: 79 tokens.
- P99: 127 tokens.
- Only 22 FAQs exceed 300 tokens.
- Only 5 FAQs exceed 500 tokens.
There is no reason to split normal FAQs.
2. Final proposed chunking strategy
I recommend a hybrid strategy rather than applying one splitter to everything.
Content	Strategy
Brief description	Single document
Details	Structure-based + Recursive
Eligibility	Structure-based + Parent-Child for long sections
Benefits	Structure-based + Recursive
Application process	Step-based splitting
Documents required	Preserve complete list
Exclusions	Single document
FAQs	Atomic Q&A

Where does semantic chunking fit?

We should test it on long details and application_process sections. It is not necessary for the majority of your structured JSON records.
3. Recommended initial configuration
Parameter	Initial value
Parent chunk size	1,200 tokens
Child chunk size	350 tokens
Child overlap	50 tokens
FAQ chunking	No splitting
Long-document threshold	800 tokens

These are experimental starting values, not final optimized settings. We should use the selected embedding model's tokenizer when implementing them.
4. Next step: Build the chunking strategy
Before implementation, one important distinction:
- Parent-Child splitting is useful when a small child chunk retrieves information but the LLM needs the larger section for context.
- Recursive splitting is useful when a long section needs straightforward size-based splitting.
- Step-based splitting is preferable when application instructions have numbered steps.

implementing structure-based Parent-Child chunking with special handling for application steps and atomic FAQs.